# Pipeline 07: SLM Distillation & Edge Export

**Goal:** Fine-tune a lightweight, open-source 1.5 Billion parameter model (Qwen 2.5) to replicate the Claude "Teacher" model's parsing capabilities, and export it for edge deployment (iOS/Mobile).

**Architecture:**
1. **Data Sanitization:** A strict validation gate filters the `.jsonl` dataset to remove any LLM hallucinations (e.g., ensuring only `tone_curve` and `color_mixer` keys exist), preventing "Garbage In, Garbage Out".
2. **QLoRA Fine-Tuning:** Leveraging the `Unsloth` library, the pipeline loads Qwen 2.5 in 4-bit precision and applies Low-Rank Adaptation (LoRA) matrices. This isolates the training updates to a tiny fraction of the neural network, allowing rapid training on a single T4 GPU.
3. **GGUF Quantization:** The finetuned adapter weights are merged back into the base model and quantized into the `.gguf` format. This compresses the model to under 1.5GB, making it capable of running entirely on-device via Apple's Neural Engine.

In [1]:
# 1. Environment Setup & Unsloth Installation
# Unsloth custom-compiles its GPU kernels, so installation takes ~3 minutes.
!pip install unsloth -q
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

PROJECT = '/content/drive/MyDrive/photo-style-rl'
DATASET_PATH = f'{PROJECT}/data/slm_training_dataset.jsonl'
OUTPUT_DIR = f'{PROJECT}/models/qwen_fuji_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Environment initialized. Ready for Unsloth.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

## Data Sanitization & Tokenization

Before feeding the synthetic dataset into the neural network, the pipeline must enforce strict schema adherence. Any JSON payload containing hallucinated keys (like "warmth" or "exposure") that our deterministic renderer cannot process is automatically purged from the training pool. The cleaned dataset is then tokenized using Qwen's specific ChatML template.

In [2]:
# 2. Data Sanitization & Loading

def sanitize_dataset(file_path):
    """Filters out any records where the Teacher LLM hallucinated bad JSON keys."""
    valid_records = []
    allowed_keys = {"tone_curve", "color_mixer"}

    with open(file_path, 'r') as f:
        for line in f:
            record = json.loads(line)
            try:
                payload = json.loads(record['messages'][1]['content'])
                is_valid = True

                # Check every region for strict schema adherence
                for region, edits in payload.items():
                    if not isinstance(edits, dict):
                        is_valid = False
                        break
                    for key in edits.keys():
                        if key not in allowed_keys:
                            is_valid = False
                            break

                if is_valid:
                    valid_records.append(record)
            except:
                pass # Skip if JSON parsing fails entirely

    return valid_records

# Load and clean the data
print("Sanitizing dataset...")
clean_data = sanitize_dataset(DATASET_PATH)
print(f"Kept {len(clean_data)} perfectly formatted records (dropped {sum(1 for _ in open(DATASET_PATH)) - len(clean_data)} hallucinations).")

# Convert to HuggingFace Dataset format
dataset = Dataset.from_list(clean_data)

# 3. Model Loading via Unsloth
print("\nLoading Qwen 2.5 1.5B in 4-bit precision...")
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True, # Shrinks the model so it fits on Colab's T4
)

# Apply Qwen's native chat template to our raw messages
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Model and Dataset ready for QLoRA.")

Sanitizing dataset...
Kept 5 perfectly formatted records (dropped 45 hallucinations).

Loading Qwen 2.5 1.5B in 4-bit precision...
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Model and Dataset ready for QLoRA.


## QLoRA Parameter-Efficient Fine-Tuning (PEFT)

To avoid catastrophic forgetting and maintain training stability, the base Qwen 2.5 model is frozen. Low-Rank Adaptation (LoRA) matrices are injected into the attention layers. The model learns to map natural language photography concepts ("cinematic," "fuji," "moody") to our strict JSON rendering schema.

In [3]:
# 4. Configure LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank of the adapter (determines how much it can learn)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Unsloth's massive memory saver
    random_state = 3407,
)

# 5. The Training Loop
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Speeds up training for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # For a 50-record Proof of Concept, 60 steps is perfect
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Disables wandb logging
    ),
)

print("Starting Fine-Tuning Engine...")
trainer_stats = trainer.train()
print(f"Training Complete. Memory used: {round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 2)} GB")

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting Fine-Tuning Engine...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,3.380016
2,3.380571
3,3.222615
4,2.853215
5,2.557701
6,2.298197
7,2.019712
8,1.766711
9,1.542258
10,1.370702


Training Complete. Memory used: 2.33 GB


## Edge Optimization (GGUF iOS Export)

The final step converts the fine-tuned PyTorch weights into the highly compressed `.gguf` format. This 4-bit quantization reduces the total model footprint to roughly 1.1 Gigabytes, allowing it to be compiled directly into Swift for local, offline execution on an iPhone via MLC LLM or MLX.

In [4]:
# 6. Export to iOS Ready Format (GGUF)
print(f"Merging LoRA adapters and compiling to GGUF (4-bit)...")

# Unsloth handles the complex GGUF conversion natively
# q4_k_m is the industry standard for mobile deployment (best balance of speed, size, and precision)
export_path = f"{OUTPUT_DIR}/qwen_fuji_q4_k_m"

model.save_pretrained_gguf(
    export_path,
    tokenizer,
    quantization_method = "q4_k_m"
)

print(f"\n✅ EDGE EXPORT SUCCESSFUL!")
print(f"Your iPhone-ready model has been saved to: {export_path}-unsloth.Q4_K_M.gguf")
print("You can now drop this file directly into LM Studio, Ollama, or an iOS MLX wrapper to run offline without internet or API keys.")

Merging LoRA adapters and compiling to GGUF (4-bit)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:42<00:00, 42.83s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:25<00:00, 85.71s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/qwen2.5-1.5b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/photo-style-rl/models/qwen_fuji_v1/qwen_fuji_q4_k_m_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/photo-